# Exercise: PC Algorithm



In this exercise, we learn a Markov equivalence class using the PC algorithm.

The first cells set up the conditional independece oracle. Run them without looking them (do not cheat by looking the correct structure in the code). The exercise starts from the cell "Exercise starts here".

### Conditional Independence Oracle


This notebook exposes a `query(X, Y, Z)` function that answers d-separation queries  
with respect to a **hidden DAG**.  

**Students:** use only the `query()` function in the *Student Interface* section below.  
> Do not read the hidden graph definition until you have completed all tasks.

---

In [ ]:
# =============================================================
# RUN THIS CELL WITHOUT LOOKING INTO IT
# =============================================================



# ============================================================
# Initialises the model 
# ============================================================

VARIABLES = ['A', 'B', 'C', 'D', 'E']

# --- Default graph --------------------
HIDDEN_EDGES1 = [
    ('A', 'C'), 
    ('B', 'C'), 
    ('C', 'D'), 
    ('C', 'E'), 
    ('D', 'E'), 
]

# --- Alternative graphs ---------------------

# Graph 2:
HIDDEN_EDGES2 = [
     ('A', 'B'),
     ('B', 'C'),
     ('B', 'D'),
     ('D', 'E'),
 ]

# Graph 3: 
HIDDEN_EDGES3 = [
     ('A', 'B'),   
     ('A', 'C'),   
     ('B', 'D'),  
     ('E', 'D'),   
     ('E', 'C'),  
 ]

# Graph 4: 
HIDDEN_EDGES4 = [
      ('A', 'B'),
     ('A', 'D'),
     ('B', 'C'),
     ('D', 'C'),
     ('D', 'E'),
     ('C', 'E'),
 ]

# ============================================================

import networkx as nx
from networkx.algorithms.d_separation import d_separated

def _build_dag(variables, edges):
    G = nx.DiGraph()
    G.add_nodes_from(variables)
    G.add_edges_from(edges)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError('HIDDEN_EDGES contains a directed cycle - not a valid DAG!')
    extra = [e for e in edges if e[0] not in variables or e[1] not in variables]
    if extra:
        raise ValueError(f'Edges reference variables not in VARIABLES: {extra}')
    return G

_HIDDEN_DAG = _build_dag(VARIABLES, HIDDEN_EDGES1)

print('The oracle is ready. You may now use query() below.')


In [ ]:
# ============================================================
#  Oracle internals — do not edit
# ============================================================

_query_log = []   # list of (X, Y, Z_list, result) tuples

def query(X, Y, Z=None):
    """
    Conditional independence oracle.

    Returns True  if X is d-separated from Y given Z in the hidden DAG.
    Returns False if X and Y are d-connected given Z.

    Parameters
    ----------
    X : str
        First variable name.
    Y : str
        Second variable name.
    Z : str | list[str] | None
        Conditioning set.  Pass [] or None for the empty set.

    Examples
    --------
    >>> query('A', 'C', [])         # Is A ⊥ C unconditionally?
    >>> query('A', 'C', 'B')        # Is A ⊥ C | B?
    >>> query('A', 'C', ['B','D'])  # Is A ⊥ C | {B, D}?
    """
    # --- Input validation ---
    if Z is None:
        Z = []
    if isinstance(Z, str):
        Z = [Z] if Z else []
    Z = list(Z)

    for v in [X, Y] + Z:
        if v not in VARIABLES:
            raise ValueError(
                f"Unknown variable '{v}'. Choose from {VARIABLES}."
            )
    if X == Y:
        raise ValueError("X and Y must be different variables.")
    if X in Z or Y in Z:
        raise ValueError("Conditioning set Z must not contain X or Y.")

    # --- D-separation test ---
    result = d_separated(_HIDDEN_DAG, {X}, {Y}, set(Z))

    # --- Log the query ---
    Z_sorted = sorted(Z)
    _query_log.append((X, Y, Z_sorted, result))

    # --- Pretty output ---
    cond_str = '{' + ', '.join(Z_sorted) + '}' if Z_sorted else '∅'
    answer   = 'INDEPENDENT (True)' if result else 'DEPENDENT   (False)'
    symbol   = '⊥' if result else '⊬'
    print(f"  {X} {symbol} {Y} | {cond_str}   →   {answer}")

    return result


def query_log():
    """Print a summary of all queries made so far."""
    if not _query_log:
        print("No queries made yet.")
        return
    print(f"{'#':<4} {'X':<4} {'Y':<4} {'Conditioning set Z':<22} {'Result'}")
    print("-" * 52)
    for i, (X, Y, Z, result) in enumerate(_query_log, 1):
        cond_str = '{' + ', '.join(Z) + '}' if Z else '∅'
        res_str  = 'indep (True)' if result else 'dep   (False)'
        print(f"{i:<4} {X:<4} {Y:<4} {cond_str:<22} {res_str}")
    print()
    print(f"Total queries: {len(_query_log)}")


def reset_log():
    """Clear the query log (e.g. to start fresh after practice runs)."""
    global _query_log
    _query_log = []
    print("Query log cleared.")


print("Oracle functions loaded:")
print("  query(X, Y, Z)  — ask a CI question")
print("  query_log()     — review all queries made")
print("  reset_log()     — clear the log")

# Exercise starts here

## Overview

In this exercise you will reconstruct the structure of an unknown Bayesian network
using only a **conditional independence oracle** — a tool that answers queries of the
form _"Is X independent of Y given the set Z?"_ with respect to a hidden directed
acyclic graph (DAG).

You will implement each phase of the **PC algorithm** by hand (on paper or in a notebook), guided by the oracle, and end up with the **CPDAG** that
represents the Markov equivalence class of the true, hidden graph.

---
## Instructions

Run everything above this line once to set up the oracle (do not peak the source code to find the correct answer). You work in the cells below.

**Your variables are:** `A`, `B`, `C`, `D`, `E`

Use `query(X, Y, Z)` to ask whether X is independent of Y given the conditioning set Z.  
Refer to the exercise sheet for the full task description.

---

### How to use the oracle

```python
query('A', 'C', [])          # Is A ⊥ C | ∅  ?
query('A', 'C', 'B')         # Is A ⊥ C | {B} ?
query('A', 'C', ['B', 'D'])  # Is A ⊥ C | {B, D} ?
```

The oracle prints a human-readable answer **and** returns a boolean you can use in code.

---
### Task 1 — Skeleton Learning

Follow the PC algorithm.  
Work through Level 0, then Level 1, then Level 2 as needed.  
Add cells as required — one cell per edge/conditioning set is a clean style.

**Tip:** keep a running note of which edges are still present and what sep sets you have found.

#### Level 0 — unconditional independence tests

In [ ]:
# Level 0: test every pair unconditionally
# There are 10 pairs for 5 variables.
# Uncomment and run each line, then decide which edges to remove.

# query('A', 'B', [])
# query('A', 'C', [])
# query('A', 'D', [])
# query('A', 'E', [])
# query('B', 'C', [])
# query('B', 'D', [])
# query('B', 'E', [])
# query('C', 'D', [])
# query('C', 'E', [])
# query('D', 'E', [])

In [ ]:
# YOUR NOTES after Level 0:
# Edges removed: ...
# Remaining skeleton edges: ...
# Separating sets found: ...

#### Level 1 — condition on single neighbours

In [ ]:
# For each remaining edge (X, Y), test X ⊥ Y | {Z}
# where Z is each neighbour of X or Y other than the other endpoint.

# Example:
# query('A', 'D', 'B')   # test A ⊥ D | {B}
# query('A', 'D', 'C')   # test A ⊥ D | {C}

# Add your queries here:

In [ ]:
# YOUR NOTES after Level 1:
# Edges removed: ...
# Remaining skeleton edges: ...
# Separating sets found: ...

#### Level 2 — condition on pairs of neighbours (if needed)

In [ ]:
# Only needed if some node still has 3+ neighbours after Level 1.
# query('X', 'Y', ['Z1', 'Z2'])

# Add your queries here (or leave blank if Level 1 was sufficient):


In [ ]:
# YOUR NOTES after Level 2:
# Edges removed: ...
# Remaining skeleton edges: ...
# Separating sets found: ...

#### Level 2 — condition on pairs of neighbours (if needed)

In [ ]:
# Only needed if some node still has 4+ neighbours after Level 2.
# query('X', 'Y', ['Z1', 'Z2'])

# Add your queries here (or leave blank if Level 2 was sufficient):

In [ ]:
# FINAL SKELETON SUMMARY
# Edges in skeleton: ...
# All separating sets:
#   sep(?, ?) = { }
#   sep(?, ?) = { }
#   ...

---
### Task 2 — Orient V-Structures

For every **unshielded triple** (X — Z — Y, where X and Y are NOT adjacent),  
check whether Z is in sep(X, Y).  
- If **Z ∉ sep(X, Y)** → orient as **X → Z ← Y** (v-structure / collider)
- If **Z ∈ sep(X, Y)** → leave both edges undirected for now

In [ ]:
# List your unshielded triples here and check each one:

# Unshielded triple (?, ?, ?):
#   sep(?, ?) = { ... }
#   Middle node ? in sep set? YES / NO
#   => Orient as: ...

# (copy this block for each unshielded triple)

In [ ]:
# V-STRUCTURES FOUND:
#   ? -> ? <- ?    (because ? ∉ sep(?, ?))
#   ...

# PDAG after this step (draw on paper or describe here):
# Directed edges  : ...
# Undirected edges: ...

---
### Task 3 — Apply Meek's Orientation Rules

Work through R1, R2, R3, R4 in order, then loop back to the top if any rule fired.

| Rule | Pattern | Action |
|------|---------|--------|
| R1 | α → β — γ,  α ⊬ γ | orient β → γ |
| R2 | α → β → γ,  α — γ | orient α → γ |
| R3 | α — β → γ,  α — δ → γ,  α — γ,  β ⊬ δ | orient α → γ |
| R4 | α — β → γ → δ,  α — δ,  α - γ | orient α → δ |

In [ ]:
# Record each rule application:

# Iteration 1:
#   R1 fires? Pattern: ... Action: orient ? -> ?
#   R2 fires? ...
#   R3 fires? ...
#   R4 fires? ...

# Iteration 2 (if any rule fired above):
#   ...

# FINAL CPDAG:
# Directed edges  : ...
# Undirected edges: ...

Count the total number of CI queries you made. Then answer:

- How many queries were at level 0? Level 1? Level 2?
- Did you make any redundant queries (i.e. queries you could have skipped)?
  If so, which ones?

(Bonus) What is the size of the Markov equivalence class?